In [1]:
%pip install pdfplumber pandas

You should consider upgrading via the '/Users/cristinasaraivafigueiredo/Documents/Documentos - Cristina’s MacBook Air/Conciliacao-extrato/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pdfplumber

with pdfplumber.open('PDF_PARA_CONCILIACAO.pdf') as pdf:
    total_paginas = len(pdf.pages)
    print(f"Total de páginas no arquivo: {total_paginas}")
    
    # Lê a primeira página
    pagina = pdf.pages[0] 
    
    print("\n--- 1. TEXTO EXTRAÍDO BRUTO ---")
    texto = pagina.extract_text()
    if texto:
        print(texto[:500])
    else:
        print("Nenhum texto extraído na página 1 (página vazia ou imagem).")
    
    print("\n--- 2. TABELAS ENCONTRADAS ---")
    tabelas = pagina.extract_tables()
    print(f"Total de tabelas encontradas: {len(tabelas)}")
    if tabelas and len(tabelas[0]) > 0:
        print("Primeira linha da tabela:", tabelas[0][0])

Total de páginas no arquivo: 2

--- 1. TEXTO EXTRAÍDO BRUTO ---
Nenhum texto extraído na página 1 (página vazia ou imagem).

--- 2. TABELAS ENCONTRADAS ---
Total de tabelas encontradas: 2
Primeira linha da tabela: ['']


In [3]:
import pdfplumber
import pandas as pd
import re

def extrair_boletos_por_posicao(caminho_pdf):
    dados_extraidos = []
    
    with pdfplumber.open(caminho_pdf) as pdf:
        for num_pag, pagina in enumerate(pdf.pages, start=1):
            # Tenta extrair o texto preservando a disposição visual da página
            texto_layout = pagina.extract_text(layout=True)
            
            # Se a extração por layout falhar, usa a extração de palavras por coordenada
            if not texto_layout or len(texto_layout.strip()) == 0:
                words = pagina.extract_words()
                # Agrupa palavras por linha aproximada (posição top)
                linhas_dict = {}
                for w in words:
                    top_aprox = round(w['top'], -1) # Agrupa linhas na mesma altura
                    linhas_dict.setdefault(top_aprox, []).append(w['text'])
                
                linhas = [" ".join(linhas_dict[k]) for k in sorted(linhas_dict.keys())]
            else:
                linhas = texto_layout.split('\n')
                
            for linha in linhas:
                linha_limpa = linha.strip()
                
                # Procura por linhas que contêm a palavra Boleto (em maiúscula ou minúscula)
                if 'boleto' in linha_limpa.lower():
                    # Captura a data (formato DD/MM/YY ou DD/MM/YYYY)
                    match_data = re.search(r'\b(\d{2}/\d{2}/\d{2,4})\b', linha_limpa)
                    
                    # Captura todos os valores monetários da linha (ex: 645,61 ou 1.375,38)
                    valores = re.findall(r'\b\d{1,3}(?:\.\d{3})*,\d{2}\b', linha_limpa)
                    
                    if match_data and valores:
                        data = match_data.group(1)
                        
                        # No extrato, se houver crédito, o valor do crédito é o penúltimo ou único valor antes do saldo
                        valor_credito = valores[-2] if len(valores) >= 2 else valores[0]
                        
                        # Isolamos a descrição (tudo entre a data e os valores)
                        desc = linha_limpa
                        
                        dados_extraidos.append({
                            'data': data,
                            'descricao': desc,
                            'credito_raw': valor_credito,
                            'pagina': num_pag
                        })

    df = pd.DataFrame(dados_extraidos)
    
    if df.empty:
        print("Ainda não foi possível extrair texto legível. O PDF pode ser uma imagem digitalizada (scanner).")
        return df
        
    # Tratamento dos números e datas
    df['credito'] = (
        df['credito_raw']
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    df['credito'] = pd.to_numeric(df['credito'], errors='coerce').fillna(0.0)
    df['data'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')
    
    return df[['data', 'descricao', 'credito', 'pagina']]

# --- EXECUÇÃO ---
df_relatorio_1 = extrair_boletos_por_posicao('PDF_PARA_CONCILIACAO.pdf')
df_relatorio_1

Ainda não foi possível extrair texto legível. O PDF pode ser uma imagem digitalizada (scanner).


""
